In [1]:
import os, sys
current_dir = os.path.abspath('')
repo_path = os.path.abspath(os.path.join(current_dir, '..', '..', 'reproduction', '2ABT_behaviour_models_updated'))
if repo_path not in sys.path:
    sys.path.append(repo_path)
from model_policies import log_likelihood_model_policy
    
phase_3_path = os.path.abspath(os.path.join('..', 'phase 3'))
if phase_3_path not in sys.path:
    sys.path.append(phase_3_path)
from nonlinear_stickiness_refined_v2 import (
    run_rflr_nonlinear_stickiness,
    run_hmm_nonlinear_stickiness,
    run_hmm_reset_nonlinear_stickiness,
    run_hmm_resetv2_nonlinear_stickiness,
    run_fq_nonlinear_stickiness,
    run_fq_value_gated_only
)

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from scipy.optimize import minimize
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns


In [2]:
data_path = os.path.join(repo_path, "mouse_data.csv")
df = pd.read_csv(data_path)
cv_results = pd.read_csv(os.path.join('..', "phase2_generative", "cv_results_fitted.csv"))

In [3]:
def extract_sessions(df, condition):
    """
    Extracts sessions for a specific condition.
    Returns:
        sessions: List of (choices, rewards) tuples for model fitting
        targets: List of true target states (0/1) for simulations
    """
    subset = df[df["Condition"] == condition]
    sessions_list = []
    targets_list = []
    
    for sess_id, group in subset.groupby("Session"):
        # Sort by trial
        group = group.sort_values("Trial")
        
        c = group["Decision"].astype(int).values
        r = group["Reward"].astype(int).values
        state = group["Target"].astype(int).values
        
        sessions_list.append((c, r))
        targets_list.append(state)
        
    return sessions_list, targets_list

In [4]:
def optimize_fold(model_type, train_sess, fixed_params):
    # Standard biological working-memory window for motor/value models
    n_steps_list = [1, 2, 3, 4, 5]
    
    # Expanded evidence-accumulation window for the cognitive model
    if model_type == "HMM_resetv2":
        n_steps_list = [1, 2, 3, 4, 5, 10, 15, 20]
        
    best_train_nll = np.inf
    best_params = None
    
    # Define bounds for (alpha_base, gamma)
    bounds = [(0.1, 5.0), (0.0, 5.0)] 

    for n in n_steps_list:
        # FQ_Value_Gated does not use gamma, so we fix it to 0.0
        if model_type == "FQ_Value_Gated":
            initial_guess = [1.5] # Only alpha_base
            fq_bounds = [(0.1, 5.0)]
            
            def objective(x):
                ab = x[0]
                # Pass 0.0 for gamma to maintain function signature compatibility
                ll = run_fq_value_gated_only(train_sess, (ab, fixed_params['k'], 1.0), n, 0.0, fit_mode=True, return_ll_only=True)
                return -ll
            
            res = minimize(objective, initial_guess, method='L-BFGS-B', bounds=fq_bounds)
            current_alpha, current_gamma = res.x[0], 0.0
        
        else:
            initial_guess = [1.5, 1.0] # alpha_base and gamma
            
            def objective(x):
                ab, g = x
                if model_type == "HMM_resetv2":
                    prms = {'q': fixed_params['q'], 'p': fixed_params['p'], 'alpha': ab, 'beta': fixed_params['beta'], 'tau': fixed_params['tau']}
                    ll = run_hmm_resetv2_nonlinear_stickiness(train_sess, prms, n, g, fit_mode=True, return_ll_only=True)
                elif model_type == "FQ":
                    ll = run_fq_nonlinear_stickiness(train_sess, (ab, fixed_params['k'], 1.0), n, g, fit_mode=True, return_ll_only=True)
                return -ll

            res = minimize(objective, initial_guess, method='L-BFGS-B', bounds=bounds)
            current_alpha, current_gamma = res.x[0], res.x[1]

        if res.fun < best_train_nll:
            best_train_nll = res.fun
            best_params = {"alpha_base": current_alpha, "gamma": current_gamma, "n_steps": n}

    return best_params

In [5]:
def _process_fold(train_idx, test_idx, sessions, model_type, fixed_params):
    train_sess = [sessions[i] for i in train_idx]
    test_sess = [sessions[i] for i in test_idx]

    best_p = optimize_fold(model_type, train_sess, fixed_params)

    if model_type == "HMM_resetv2":
        prms = {'q': fixed_params['q'], 'p': fixed_params['p'], 'alpha': best_p["alpha_base"], 
                'beta': fixed_params['beta'], 'tau': fixed_params['tau']}
        ll = run_hmm_resetv2_nonlinear_stickiness(
            test_sess, prms, best_p["n_steps"], best_p["gamma"], 
            fit_mode=True, return_ll_only=True
        )
    elif model_type == "FQ":
        ll = run_fq_nonlinear_stickiness(
            test_sess, 
            (best_p["alpha_base"], fixed_params['k'], fixed_params['temp']), 
            best_p["n_steps"], 
            0.0, 
            fit_mode=True, 
            return_ll_only=True
        )
    
    return {"ll": ll, "params": best_p}

In [6]:
def cross_validate_model(model_type, sessions, fixed_params, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    results = Parallel(n_jobs=-1)(
        delayed(_process_fold)(train_idx, test_idx, sessions, model_type, fixed_params)
        for train_idx, test_idx in kf.split(sessions)
    )

    test_lls = [res["ll"] for res in results]
    fold_params = [res["params"] for res in results]

    avg_alpha = np.mean([p["alpha_base"] for p in fold_params])
    std_alpha = np.std([p["alpha_base"] for p in fold_params])
    
    avg_gamma = np.mean([p["gamma"] for p in fold_params])
    std_gamma = np.std([p["gamma"] for p in fold_params])

    return {
        "Total_Test_LL": np.sum(test_lls),
        "Avg_Alpha": avg_alpha,
        "Std_Alpha": std_alpha,   # Add to return dictionary
        "Avg_Gamma": avg_gamma,
        "Std_Gamma": std_gamma,   # Add to return dictionary
        "Best_N_Steps": int(np.bincount([p["n_steps"] for p in fold_params]).argmax()) 
    }

In [7]:
def fit_condition_cv(condition):
    print(f"\n{'='*55}")
    print(f"--- Fitting Condition: {condition} (5-Fold CV) ---")
    print(f"{'='*55}")
    
    # 1. Data Preparation
    sessions, targets = extract_sessions(df, condition)
    total_trials = sum(len(c) for c, r in sessions)
    
    # Calculate HMM Hazard Rate (q) from block lengths
    est_p_switch = 1.0 / np.mean([len(t)/np.sum(np.abs(np.diff(t))) for t in targets])
    q_val = 1.0 - est_p_switch

    # 2. Extract Phase 2 "Hardware" Baselines (Fixed)
    beta_val = cv_results[cv_results["Condition"] == condition]["Beta"].mean()
    tau_val = cv_results[cv_results["Condition"] == condition]["Tau"].mean()
    k_val = 1 - np.exp(-1 / tau_val)
    p_val = int(condition.split("-")[0]) / 100.0

    r_params = {'beta': beta_val, 'tau': tau_val} 
    h_params = {'q': q_val, 'p': p_val, **r_params} 
    f_params = {'k': k_val, 'temp': 1.0}

    # 3. Parallel Optimization Execution
    print(f">> Dataset Size: {len(sessions)} sessions, {total_trials} total trials")
    
    results = {}
    
    model_configs = [
    ("HMM_resetv2", h_params),  
    ("FQ", f_params)
]

    for name, params in model_configs:
        print(f">> Optimizing {name} (Parallel)...")
        res = cross_validate_model(name, sessions, params)
        
        # Calculate Mean LL for Phase 2 comparison
        mLL = res['Total_Test_LL'] / total_trials
        
        print(f"{name:4} | Mean LL : {mLL:.4f}")
        print(f"{name:4} | Params  : a_base={res['Avg_Alpha']:.3f} (±{res['Std_Alpha']:.2f}), "
              f"gamma={res['Avg_Gamma']:.3f} (±{res['Std_Gamma']:.2f}), N={res['Best_N_Steps']}")
        print("-" * 30)
        
        results[name] = res

    return results

In [8]:
final_results_v3 = {}
for cond in ["80-20", "70-30"]:
    final_results_v3[cond] = fit_condition_cv(cond)


--- Fitting Condition: 80-20 (5-Fold CV) ---
>> Dataset Size: 159 sessions, 117334 total trials
>> Optimizing HMM_resetv2 (Parallel)...
HMM_resetv2 | Mean LL : -0.1832
HMM_resetv2 | Params  : a_base=0.809 (±0.01), gamma=0.661 (±0.01), N=1
------------------------------
>> Optimizing FQ (Parallel)...
FQ   | Mean LL : -0.2204
FQ   | Params  : a_base=2.076 (±0.01), gamma=0.002 (±0.00), N=2
------------------------------

--- Fitting Condition: 70-30 (5-Fold CV) ---
>> Dataset Size: 200 sessions, 154872 total trials
>> Optimizing HMM_resetv2 (Parallel)...
HMM_resetv2 | Mean LL : -0.2015
HMM_resetv2 | Params  : a_base=0.997 (±0.01), gamma=0.561 (±0.01), N=1
------------------------------
>> Optimizing FQ (Parallel)...
FQ   | Mean LL : -0.2356
FQ   | Params  : a_base=1.702 (±0.01), gamma=0.548 (±0.02), N=2
------------------------------
